# 01 — Ingest + Clean (PRODUCTION PIPELINE)

**What this notebook is:** the actual reusable IBP-cycle pipeline — the part that gets deployed and scheduled in production. Its logic lives entirely in `src/`; this notebook only orchestrates it.

**Its only assumption about the input:** raw CSVs matching `schema.raw.*` exist at `<data_root>/raw/`. It does not care whether they came from `generate_data.py` (this capstone) or a real ERP extract (production) — nothing in `src/ingest.py` or `src/cleaner.py` references the generator.

**Prerequisite:** run `00_generate_test_data.ipynb` first and have `ibp_raw_data.zip` ready to upload. This can be a completely separate Colab session/runtime — no session-sharing assumptions.

**Output — the pipeline's real deliverables:**
- `<data_root>/clean/clean_master.parquet` — the canonical clean table
- `<data_root>/clean/dq_report.csv` / `.md` — the data quality report
- `<data_root>/clean/Step4_Data_Quality_Review.xlsx` — the human sign-off artefact. **Step 5 does not begin until this workbook is reviewed and signed off.**

## Setup — clone the repo (for `src/`, `config/`, `tests/` — code only, no data)

In [ ]:
import subprocess, os, sys

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])
    return r

REPO = '/content/ibp-tradeoff'
os.chdir('/content')
sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO)
sys.path.insert(0, REPO)
print('Working directory:', os.getcwd())


## Upload the raw data
Run this cell, then click **Choose Files** and select `ibp_raw_data.zip` (downloaded from notebook 00). This is the explicit, traceable hand-off point between the scaffolding step and the production pipeline.

In [ ]:
from google.colab import files
import zipfile, io

print('Select ibp_raw_data.zip from your computer:')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
    z.extractall(REPO)

for tag in ['data_primary', 'data_control']:
    raw_dir = os.path.join(REPO, tag, 'raw')
    if not os.path.isdir(raw_dir):
        raise FileNotFoundError(
            f"{raw_dir} not found after extracting {zip_name}. "
            "Confirm you uploaded ibp_raw_data.zip from notebook 00, not a different file."
        )
    n_files = len(os.listdir(raw_dir))
    print(f'{tag}/raw: {n_files} files found')

print()
print('Raw data loaded successfully. Proceeding to ingest + clean.')


## Ingest + clean both datasets
Uses `src/ingest.py` (`DataIngestor`), `src/cleaner.py` (`DataCleaner`), and `src/report_step4.py` (`build_review_workbook`). Same code, both datasets, zero edits — this is the reusable-pipeline evidence.

Expect on both: rows out **2,159** · skus **60** · nulls **87** · recovered **86** · roll-forward max **0.000007**

In [ ]:
import pandas as pd
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
from src.report_step4 import build_review_workbook

results = {}
dq_rows = []

for data_root in ['data_primary', 'data_control']:
    print('=' * 78)
    print(f'DATASET: {data_root}')
    print('=' * 78)

    ingestor = DataIngestor(repo_root=REPO, data_root=data_root)
    raw = ingestor.load()

    cleaner = DataCleaner(ingestor.schema, ingestor.assumptions)
    df_clean, sku_master_clean, dq_report = cleaner.clean(raw)

    rows_in = sum(v for k, v in ingestor.rows_in_by_source.items() if k != 'sku_master')
    print('\nRECONCILIATION')
    for source, n in ingestor.rows_in_by_source.items():
        print(f'  rows in  . {source:<12} {n:>6,}')
    print(f'  rows in  . transactional total   {rows_in:>6,}')
    for d in cleaner.dropped:
        print(f"  dropped  . {d['reason']:<45} {d['rows']:>3}")
    print(f'  rows out . clean_master          {len(df_clean):>6,}')

    print('\nDATA QUALITY REPORT')
    print(dq_report.to_string(index=False))

    out_dir = os.path.join(REPO, data_root, 'clean')
    os.makedirs(out_dir, exist_ok=True)

    parquet_path = os.path.join(out_dir, 'clean_master.parquet')
    df_clean.to_parquet(parquet_path, index=False)

    # D-043: sku_master is persisted, not just held in memory. Step 6's engine
    # needs price_eur, std_cost_eur, gross_margin_eur, shelf_life_days,
    # moq_units, min_run_units and line_code per SKU, and Step 10's Streamlit
    # app cannot re-run the cleaner to get them. Persisting reviewed content;
    # no cleaning logic changes, so no re-sign-off.
    sku_master_path = os.path.join(out_dir, 'sku_master.parquet')
    sku_master_clean.to_parquet(sku_master_path, index=False)

    dq_csv_path = os.path.join(out_dir, 'dq_report.csv')
    dq_md_path = os.path.join(out_dir, 'dq_report.md')
    dq_report.to_csv(dq_csv_path, index=False)
    with open(dq_md_path, 'w') as f:
        f.write(f'# Data Quality Report — {data_root}\n\n')
        f.write('Generated by `src/cleaner.py` (`DataCleaner`), Step 4.\n')
        f.write('Implements `cleaning-spec.md` C-01 to C-13.\n\n')
        f.write(dq_report.to_markdown(index=False))
        f.write('\n')

    xlsx_path = os.path.join(out_dir, 'Step4_Data_Quality_Review.xlsx')
    build_review_workbook(
        raw=raw, df_clean=df_clean, master=sku_master_clean, dq_report=dq_report,
        dropped=cleaner.dropped, out_path=xlsx_path, dataset_label=data_root,
    )

    print(f'\nwrote {parquet_path}')
    print(f'wrote {sku_master_path}')
    print(f'wrote {dq_csv_path}')
    print(f'wrote {dq_md_path}')
    print(f'wrote {xlsx_path}')

    dq_tagged = dq_report.copy()
    dq_tagged.insert(0, 'dataset', data_root)
    dq_rows.append(dq_tagged)
    results[data_root] = (df_clean, sku_master_clean, dq_report)

combined_dq = pd.concat(dq_rows, ignore_index=True)
combined_path = os.path.join(REPO, 'data_quality_summary.csv')
combined_dq.to_csv(combined_path, index=False)
print(f'\nwrote {combined_path}')

print('\n' + '=' * 78)
print('REUSABILITY CHECK — same DataCleaner, two datasets, no code change')
for tag, (df, _, dq) in results.items():
    d = dict(zip(dq.metric, dq.value))
    print(
        f"  {tag:<14} rows={int(d['rows_out']):,}  skus={int(d['skus_out'])}  "
        f"nulls={int(d['nulls_in_volume'])}  recovered={int(d['nulls_recovered_by_identity'])}  "
        f"rollforward_max={d['stock_rollforward_max_abs_diff']:.6f}"
    )

print()
print('>>> ACTION REQUIRED before Step 5:')
print('>>> Download data_primary/clean/Step4_Data_Quality_Review.xlsx (Colab file browser),')
print('>>> review it, and complete the sign-off on sheet 1.')

df_clean = results['data_primary'][0]


## Get the review workbook via Google Drive

`files.download()` is stuck / not completing in your environment (confirmed twice) -- this is a known failure mode when a network policy or security software blocks Colab's direct browser-download mechanism. Common on corporate networks.

This works around it: mount your Google Drive, copy the files there. **You still choose the destination folder** -- just one step later, when you download from drive.google.com to your computer using Drive's own download button, which is a normal HTTPS download rather than Colab's blocked mechanism.

Running this cell asks you to authorize Colab to access your Drive (a genuine Google sign-in popup -- allow it). Files land in **My Drive / ibp-tradeoff-outputs /**

**STOP after this cell.** Go to drive.google.com, open that folder, download Step4_Data_Quality_Review.xlsx to whichever folder on your computer you choose, review it, and complete the sign-off on sheet 1 before returning here.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
out_dir = "/content/drive/My Drive/ibp-tradeoff-outputs"
os.makedirs(out_dir, exist_ok=True)

shutil.copy(
    "/content/ibp-tradeoff/data_primary/clean/Step4_Data_Quality_Review.xlsx",
    os.path.join(out_dir, "Step4_Data_Quality_Review.xlsx"),
)
shutil.copy(
    "/content/ibp-tradeoff/data_primary/clean/clean_master.parquet",
    os.path.join(out_dir, "clean_master.parquet"),
)

print(f"Copied to Google Drive: {out_dir}")
print("Go to drive.google.com, open ibp-tradeoff-outputs, and download from there to any folder you choose.")


## Gate: upload your SIGNED workbook to proceed

Do not continue until you have opened Step4_Data_Quality_Review.xlsx yourself, read it, and filled in the sign-off block on sheet 1 (Reviewed by / Date / Decision). Upload the SIGNED file below -- not the one downloaded in the previous cell.

The next cell reads the Decision cell programmatically and will refuse to produce any further output unless it says Approved or Approved with comments. This enforces the approval boundary in code, not just by convention.

In [ ]:
from google.colab import files as _files
print("Select your SIGNED Step4_Data_Quality_Review.xlsx:")
_signed = _files.upload()
signed_name = list(_signed.keys())[0]
with open(signed_name, "wb") as f:
    f.write(_signed[signed_name])
print(f"Uploaded: {signed_name}")


In [ ]:
from openpyxl import load_workbook

wb = load_workbook(signed_name, data_only=True)
ws = wb["1. Review & Approval"]

# Find the Decision row by label, rather than a hard-coded cell reference --
# the sign-off block position could shift if the workbook layout changes.
decision_value = None
reviewed_by = None
for row in ws.iter_rows(min_col=2, max_col=2):
    for cell in row:
        if cell.value == "Decision":
            decision_value = ws.cell(row=cell.row, column=3).value
        if cell.value == "Reviewed by":
            reviewed_by = ws.cell(row=cell.row, column=3).value

print(f"Reviewed by: {reviewed_by}")
print(f"Decision:    {decision_value}")

APPROVED_VALUES = {"Approved", "Approved with comments"}
if decision_value not in APPROVED_VALUES:
    raise RuntimeError(
        f"Step 4 is NOT approved (Decision = {decision_value!r}). "
        "Step 5 cannot proceed. Resolve the review comments, re-sign, and re-upload "
        "before running the next cell."
    )

print()
print(f">>> Step 4 APPROVED by {reviewed_by}. Proceeding to Step 5 export.")


## Step 5 export -- only reachable after the gate above passes
Produces the blind-estimator CSV: only the columns a blind analyst should see, no category/abc_class, nothing from sku_master that could hint at structure. This file did not exist until Step 4 was confirmed approved by the cell above.

In [ ]:
from google.colab import files

blind_cols = ["sku_id", "month", "actual_units", "forecast_l1_units",
              "forecast_l2_units", "forecast_l3_units", "production_units",
              "stock_close_units", "stock_open_units", "sched_adherence",
              "yield_rate", "promo_flag"]
blind_export = df_clean[blind_cols].sort_values(["sku_id", "month"])
blind_path = "/content/ibp-tradeoff/data_primary/clean/clean_master_blind.csv"
blind_export.to_csv(blind_path, index=False)
print(f"wrote {blind_path}  ({len(blind_export)} rows, {blind_export.sku_id.nunique()} SKUs)")
files.download(blind_path)


## (Optional) Run the regression + robustness test suite

In [ ]:
sh('pip install pytest -q')
sh('python -m pytest tests/ -v', cwd=REPO)
